# OushudhCheck: YOLO26s Blister-Strip Detector Training

Trains a real (`medicine_strip`) YOLO26s object detector** and exports a static FP32 ONNX file (`yolo_strips.onnx`) verified on pure `onnxruntime` CPU.

*(Recommended)* In Kaggle **Add Input**, add 1–2 public blister-pack / medicine-strip YOLO datasets (for example: *The Drug Name Detection Dataset*, *Medicine Tablet Pack Image Dataset*, or a Roboflow YOLO export ZIP/folder of *BlisterPack and Pills* / *Medicine Strips*). **Even if you add zero Kaggle datasets, Cell 3 automatically generates 3,000 realistic multi-strip reflective Bangladeshi foil-back tabletop scenes** so the notebook always runs end-to-end!'

Download `/kaggle/working/yolo_strips.onnx` (and optional `yolo26n_strips.onnx` backup) and copy it into `D:\Projects\OushudhCheck\data\processed\yolo_strips.onnx`.

In [ ]:
!pip install -q -U "ultralytics>=8.4.0" onnx onnxslim onnxruntime pillow opencv-python-headless

import torch, ultralytics, onnxruntime as ort
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__} | GPUs: {torch.cuda.device_count()} | Ultralytics: {ultralytics.__version__} | ORT: {ort.__version__}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

_probe = YOLO("yolo26s.pt")
assert _probe.task == "detect", f"Expected task='detect', got {_probe.task!r}"
print("Verified yolo26s.pt task == 'detect'!")
del _probe

In [ ]:
from pathlib import Path
import hashlib
import random
import shutil
import cv2
import numpy as np

WORK_DS = Path("/kaggle/working/blister_dataset")
for split in ("train", "val"):
    (WORK_DS / "images" / split).mkdir(parents=True, exist_ok=True)
    (WORK_DS / "labels" / split).mkdir(parents=True, exist_ok=True)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
MIN_MEDIAN_BOX_AREA = 0.015
seen_hashes = set()
public_pairs = []

def parse_and_sanitize_label(lbl_path: Path):
    """Remap all valid whole-strip bbox/polygon annotations to class 0: medicine_strip."""
    boxes = []
    try:
        text = lbl_path.read_text(encoding="utf-8", errors="ignore")
    except OSError:
        return []
    for raw in text.splitlines():
        parts = raw.strip().split()
        if len(parts) < 5:
            continue
        try:
            vals = [float(x) for x in parts[1:]]
        except ValueError:
            continue
        if len(vals) == 4:
            xc, yc, bw, bh = vals
        elif len(vals) >= 6 and len(vals) % 2 == 0:
            # Polygon segmentation -> tight axis-aligned bbox
            xs = vals[0::2]
            ys = vals[1::2]
            x1, x2 = max(0.0, min(xs)), min(1.0, max(xs))
            y1, y2 = max(0.0, min(ys)), min(1.0, max(ys))
            bw, bh = x2 - x1, y2 - y1
            xc, yc = x1 + bw / 2.0, y1 + bh / 2.0
        else:
            continue
        if bw <= 0.02 or bh <= 0.02 or bw > 1.0 or bh > 1.0:
            continue
        xc = min(1.0, max(0.0, xc))
        yc = min(1.0, max(0.0, yc))
        boxes.append((0, xc, yc, bw, bh))
    if not boxes:
        return []
    median_area = float(np.median([b[3] * b[4] for b in boxes]))
    if median_area < MIN_MEDIAN_BOX_AREA:
        return []
    return boxes

input_root = Path("/kaggle/input")
if input_root.exists():
    for img_path in input_root.rglob("*"):
        if img_path.suffix.lower() not in IMG_EXTS:
            continue
        # Locate corresponding YOLO .txt label
        candidates = [
            img_path.with_suffix(".txt"),
            Path(str(img_path).replace("/images/", "/labels/")).with_suffix(".txt"),
        ]
        lbl_path = next((p for p in candidates if p.exists()), None)
        if lbl_path is None:
            continue
        boxes = parse_and_sanitize_label(lbl_path)
        if not boxes:
            continue
        h = hashlib.md5(img_path.read_bytes()).hexdigest()
        if h in seen_hashes:
            continue
        seen_hashes.add(h)
        public_pairs.append((img_path, boxes))

random.Random(42).shuffle(public_pairs)
n_val_pub = int(len(public_pairs) * 0.15)
for idx, (img_path, boxes) in enumerate(public_pairs):
    split = "val" if idx < n_val_pub else "train"
    stem = f"pub_{idx:05d}"
    shutil.copy2(img_path, WORK_DS / "images" / split / f"{stem}{img_path.suffix.lower()}")
    lbl_lines = [f"{c} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}" for c, xc, yc, bw, bh in boxes]
    (WORK_DS / "labels" / split / f"{stem}.txt").write_text("\n".join(lbl_lines), encoding="utf-8")

print(f"Collected {len(public_pairs)} sanitized public whole-strip images (train={len(public_pairs)-n_val_pub}, val={n_val_pub}).")

In [ ]:
# Procedural Bangladeshi Metallic Back-Foil & Multi-Strip Tabletop Synthesizer
# Bridges the public front-bubble domain gap to real Bangladeshi silver/white aluminum back-foil strips
# (including 1-6 strips per photo, torn edges, 0-360 deg rotation, drop shadows, specular glare, and hard negatives).
import math

NUM_SYNTH_TRAIN = 2600 if len(public_pairs) < 1500 else 1600
NUM_SYNTH_VAL = 350
HARD_NEG_RATIO = 0.10  # 10% empty tabletop/glare images to suppress desk false positives

BD_BRANDS = [
    "Napa Extra", "Seclo 20", "Sergel 20", "Amdocal 5", "Tufnil 200", "Diazepam 5",
    "Vitamin B-Complex", "Emistat 8", "Fluver 10", "Seronex 50", "Epitra 1", "Prelica 50",
    "Naprosyn 500", "Orthogen", "Monas 10", "Fexo 120", "Pantonix 20", "Maxpro 20",
    "Alatrol 10", "Bizoran 5/20", "Ciprocin 500", "Zimax 500", "Cef-3 200", "Rivotril 0.5",
]
BD_SUBS = [
    "Paracetamol 500 mg + Caffeine 65 mg", "Esomeprazole 20 mg", "Amlodipine 5 mg",
    "Tolfenamic Acid 200 mg", "Diazepam BP 5 mg", "Ondansetron 8 mg", "Flunarizine 10 mg",
    "Tablet USP | Beximco Pharma", "Square Pharmaceuticals Ltd.", "Incepta Pharma",
    "Essential Drugs Co. Ltd. (EDCL)", "Eskayef Pharmaceuticals", "ACI Limited",
]

def make_tabletop_bg(rng: np.random.Generator, w: int, h: int) -> np.ndarray:
    style = int(rng.integers(0, 5))
    if style == 0:  # Warm wood grain
        base = rng.integers(85, 165, size=3, dtype=np.int32)
        bg = np.full((h, w, 3), base, dtype=np.float32)
        stripes = np.sin(np.linspace(0, rng.uniform(12, 35), h))[:, None, None] * rng.uniform(6, 18)
        bg += stripes
    elif style == 1:  # Glass / glossy cool desk
        base = rng.integers(150, 235, size=3, dtype=np.int32)
        bg = np.full((h, w, 3), base, dtype=np.float32)
    elif style == 2:  # Dark matte desk
        base = rng.integers(25, 80, size=3, dtype=np.int32)
        bg = np.full((h, w, 3), base, dtype=np.float32)
    elif style == 3:  # Bedsheet / fabric pattern
        base = rng.integers(120, 220, size=3, dtype=np.int32)
        bg = np.full((h, w, 3), base, dtype=np.float32)
        gx = np.sin(np.linspace(0, 40, w))[None, :, None] * 10
        gy = np.sin(np.linspace(0, 40, h))[:, None, None] * 10
        bg += gx + gy
    else:  # Tile / marble
        base = rng.integers(170, 240, size=3, dtype=np.int32)
        bg = np.full((h, w, 3), base, dtype=np.float32)
        cv2.line(bg, (0, int(rng.integers(0, h))), (w, int(rng.integers(0, h))), (120, 120, 120), 2)
    # Add subtle lighting gradient + sensor noise
    grad = np.linspace(rng.uniform(-22, 0), rng.uniform(0, 22), w, dtype=np.float32)[None, :, None]
    bg = bg + grad + rng.normal(0, 4.5, size=(h, w, 3)).astype(np.float32)
    return np.clip(bg, 0, 255).astype(np.uint8)

def make_foil_strip(rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    """Synthesize a reflective silver/white/tinted blister foil strip + alpha mask (optionally torn)."""
    sw = int(rng.integers(180, 360))
    sh = int(rng.integers(75, 165))
    foil_type = int(rng.integers(0, 4))
    if foil_type == 0:  # Silver metallic aluminum
        color = np.array([rng.integers(185, 225)] * 3, dtype=np.float32)
        ink = (int(rng.integers(15, 70)), int(rng.integers(15, 85)), int(rng.integers(15, 120)))
    elif foil_type == 1:  # Matte white / EDCL paper-foil
        color = np.array([232, 236, 238], dtype=np.float32)
        ink = (int(rng.integers(15, 60)), int(rng.integers(60, 140)), int(rng.integers(15, 60)))
    elif foil_type == 2:  # Pale tinted blue/green foil
        color = np.array([rng.integers(190, 230), rng.integers(200, 235), rng.integers(175, 215)], dtype=np.float32)
        ink = (40, 30, 130)
    else:  # Gold / warm metallic foil
        color = np.array([rng.integers(135, 175), rng.integers(185, 215), rng.integers(205, 235)], dtype=np.float32)
        ink = (30, 30, 50)

    strip = np.full((sh, sw, 3), color, dtype=np.float32)
    # Brushed foil texture + pill cavity emboss circles
    strip += rng.normal(0, 5.0, size=(sh, sw, 3)).astype(np.float32)
    strip = np.clip(strip, 0, 255).astype(np.uint8)

    rows = int(rng.integers(1, 3))
    cols = int(rng.integers(3, 6))
    for r in range(rows):
        for c in range(cols):
            cx = int((c + 0.5) * sw / cols)
            cy = int((r + 0.5) * sh / rows)
            rad = max(8, min(sw // (cols * 3), sh // (rows * 3)))
            cv2.circle(strip, (cx, cy), rad, tuple(int(max(0, v - 18)) for v in color), 1, cv2.LINE_AA)

    # Repeated pharmaceutical brand + generic text lines
    brand = BD_BRANDS[int(rng.integers(0, len(BD_BRANDS)))]
    sub = BD_SUBS[int(rng.integers(0, len(BD_SUBS)))]
    y_cursor = 24
    while y_cursor < sh - 8:
        x_off = int(rng.integers(6, 26))
        cv2.putText(strip, brand, (x_off, y_cursor), cv2.FONT_HERSHEY_SIMPLEX, rng.uniform(0.45, 0.72), ink, 1, cv2.LINE_AA)
        cv2.putText(strip, sub[:26], (x_off, y_cursor + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.34, ink, 1, cv2.LINE_AA)
        y_cursor += int(rng.integers(30, 44))

    # Specular foil glare streak
    if rng.random() < 0.7:
        glare = np.zeros((sh, sw), dtype=np.float32)
        p1 = (int(rng.integers(0, sw)), 0)
        p2 = (int(rng.integers(0, sw)), sh)
        cv2.line(glare, p1, p2, 1.0, int(rng.integers(14, 42)))
        glare = cv2.GaussianBlur(glare, (0, 0), 14.0)
        strip = np.clip(strip.astype(np.float32) + glare[:, :, None] * rng.uniform(45, 95), 0, 255).astype(np.uint8)

    # Crimped foil border + optional torn corner mask
    cv2.rectangle(strip, (1, 1), (sw - 2, sh - 2), (140, 140, 140), 2)
    mask = np.full((sh, sw), 255, dtype=np.uint8)
    if rng.random() < 0.35:  # Simulate torn half-used strip edge
        cut_x = int(rng.uniform(0.55, 0.88) * sw)
        pts = np.array([[cut_x, 0], [sw, 0], [sw, sh], [max(0, cut_x + int(rng.integers(-25, 25))), sh]], dtype=np.int32)
        cv2.fillPoly(mask, [pts], 0)

    return strip, mask

def rotate_rgba(img: np.ndarray, mask: np.ndarray, angle: float) -> tuple[np.ndarray, np.ndarray]:
    h, w = img.shape[:2]
    rad = math.radians(angle)
    cos_a, sin_a = abs(math.cos(rad)), abs(math.sin(rad))
    nw = max(1, int(round(h * sin_a + w * cos_a)))
    nh = max(1, int(round(h * cos_a + w * sin_a)))
    M = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, 1.0)
    M[0, 2] += (nw - w) / 2.0
    M[1, 2] += (nh - h) / 2.0
    r_img = cv2.warpAffine(img, M, (nw, nh), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    r_mask = cv2.warpAffine(mask, M, (nw, nh), flags=cv2.INTER_NEAREST, borderValue=0)
    return r_img, r_mask

def generate_synthetic_split(split: str, count: int, seed: int):
    rng = np.random.default_rng(seed)
    for idx in range(count):
        cw, ch = (960, 720) if rng.random() < 0.6 else (720, 960)
        canvas = make_tabletop_bg(rng, cw, ch)
        labels = []

        if rng.random() >= HARD_NEG_RATIO:
            n_strips = int(rng.choice([1, 2, 3, 4, 5, 6], p=[0.18, 0.22, 0.20, 0.18, 0.14, 0.08]))
            placed = []
            for _ in range(n_strips):
                strip, mask = make_foil_strip(rng)
                angle = float(rng.choice([0, 90, 180, 270])) + float(rng.uniform(-22, 22))
                r_img, r_mask = rotate_rgba(strip, mask, angle)
                rh, rw = r_img.shape[:2]
                if rw >= cw - 12 or rh >= ch - 12:
                    sc = min((cw - 24) / rw, (ch - 24) / rh)
                    rw, rh = max(24, int(rw * sc)), max(24, int(rh * sc))
                    r_img = cv2.resize(r_img, (rw, rh))
                    r_mask = cv2.resize(r_mask, (rw, rh), interpolation=cv2.INTER_NEAREST)

                ok = False
                for _attempt in range(18):
                    px = int(rng.integers(6, max(7, cw - rw - 6)))
                    py = int(rng.integers(6, max(7, ch - rh - 6)))
                    cand = (px, py, px + rw, py + rh)
                    # Allow occasional mild overlap (IoU <= 0.12)
                    overlap_bad = False
                    for ox1, oy1, ox2, oy2 in placed:
                        ix = max(0, min(cand[2], ox2) - max(cand[0], ox1))
                        iy = max(0, min(cand[3], oy2) - max(cand[1], oy1))
                        inter = ix * iy
                        union = rw * rh + (ox2 - ox1) * (oy2 - oy1) - inter
                        if union > 0 and (inter / union) > 0.12:
                            overlap_bad = True
                            break
                    if not overlap_bad:
                        ok = True
                        break
                if not ok:
                    continue

                # Soft drop shadow + edge-feathered alpha composite (kills paste-seam shortcuts)
                sx, sy = min(cw - rw, px + 5), min(ch - rh, py + 6)
                sh_alpha = (cv2.GaussianBlur(r_mask, (9, 9), 3.0).astype(np.float32) / 255.0) * 0.32
                roi_sh = canvas[sy:sy+rh, sx:sx+rw].astype(np.float32)
                canvas[sy:sy+rh, sx:sx+rw] = np.clip(roi_sh * (1.0 - sh_alpha[:, :, None]), 0, 255).astype(np.uint8)

                alpha = cv2.GaussianBlur(r_mask, (3, 3), 0.7).astype(np.float32) / 255.0
                roi = canvas[py:py+rh, px:px+rw].astype(np.float32)
                blended = r_img.astype(np.float32) * alpha[:, :, None] + roi * (1.0 - alpha[:, :, None])
                canvas[py:py+rh, px:px+rw] = np.clip(blended, 0, 255).astype(np.uint8)

                ys, xs = np.where(r_mask > 127)
                if len(xs) < 20:
                    continue
                bx1, bx2 = px + int(xs.min()), px + int(xs.max())
                by1, by2 = py + int(ys.min()), py + int(ys.max())
                placed.append((bx1, by1, bx2, by2))
                xc = ((bx1 + bx2) / 2.0) / cw
                yc = ((by1 + by2) / 2.0) / ch
                bw = (bx2 - bx1) / cw
                bh = (by2 - by1) / ch
                labels.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

        # Post-composite global camera degradation (shared blur + JPEG compression)
        if rng.random() < 0.35:
            canvas = cv2.GaussianBlur(canvas, (3, 3), float(rng.uniform(0.3, 0.9)))
        stem = f"syn_{split}_{idx:05d}"
        cv2.imwrite(str(WORK_DS / "images" / split / f"{stem}.jpg"), canvas, [int(cv2.IMWRITE_JPEG_QUALITY), int(rng.integers(72, 95))])
        (WORK_DS / "labels" / split / f"{stem}.txt").write_text("\n".join(labels), encoding="utf-8")

generate_synthetic_split("train", NUM_SYNTH_TRAIN, seed=101)
generate_synthetic_split("val", NUM_SYNTH_VAL, seed=202)

yaml_text = f"""path: {WORK_DS}
train: images/train
val: images/val
names:
  0: medicine_strip
"""
(WORK_DS / "data.yaml").write_text(yaml_text, encoding="utf-8")
n_tr = len(list((WORK_DS / "images" / "train").glob("*")))
n_va = len(list((WORK_DS / "images" / "val").glob("*")))
print(f"Dataset ready -> train={n_tr}, val={n_va} | YAML: {WORK_DS / 'data.yaml'}")

In [ ]:
# Train Yolo
import torch

DEVICES = "0,1" if torch.cuda.device_count() >= 2 else "0"
print(f"Launching YOLO26s training on device={DEVICES}...")

!yolo detect train \
  model=yolo26s.pt \
  data=/kaggle/working/blister_dataset/data.yaml \
  imgsz=640 \
  epochs=50 \
  patience=12 \
  batch=32 \
  device=$DEVICES \
  workers=2 \
  amp=True \
  cos_lr=True \
  mosaic=1.0 \
  close_mosaic=10 \
  degrees=15.0 \
  flipud=0.5 \
  fliplr=0.5 \
  hsv_v=0.4 \
  hsv_s=0.5 \
  scale=0.5 \
  project=/kaggle/working/runs \
  name=yolo26s_strips \
  exist_ok=True

In [ ]:
# Export static FP32 ONNX (opset=17, simplify=True) + Pure ONNXRuntime Parity Test
from pathlib import Path
import shutil
import cv2
import numpy as np
import onnx
import onnxruntime as ort
from ultralytics import YOLO
from IPython.display import FileLink, display

best_pt = Path("/kaggle/working/runs/yolo26s_strips/weights/best.pt")
assert best_pt.exists(), f"Trained weights not found at {best_pt}"

model = YOLO(str(best_pt))
metrics = model.val(data="/kaggle/working/blister_dataset/data.yaml", imgsz=640)
print(f"Val mAP50={metrics.box.map50:.4f} | mAP50-95={metrics.box.map:.4f} | Recall={metrics.box.mr:.4f} | Precision={metrics.box.mp:.4f}")

exported_path = Path(
    model.export(
        format="onnx",
        imgsz=640,
        opset=17,
        simplify=True,
        dynamic=False,
        half=False,
    )
)
out_onnx = Path("/kaggle/working/yolo_strips.onnx")
shutil.copy2(exported_path, out_onnx)
onnx.checker.check_model(str(out_onnx))

sess = ort.InferenceSession(str(out_onnx), providers=["CPUExecutionProvider"])
in_meta = sess.get_inputs()[0]
out_meta = sess.get_outputs()[0]
print(f"ONNX Ready: {out_onnx} ({out_onnx.stat().st_size / (1024*1024):.2f} MB)")
print(f"  Input : {in_meta.name} {in_meta.shape} {in_meta.type}")
print(f"  Output: {out_meta.name} {out_meta.shape} {out_meta.type}")

val_imgs = sorted((Path("/kaggle/working/blister_dataset/images/val")).glob("*.jpg"))[:5]
for vp in val_imgs:
    img = cv2.imread(str(vp))
    h, w = img.shape[:2]
    sc = min(640.0 / h, 640.0 / w)
    nw, nh = int(round(w * sc)), int(round(h * sc))
    res = cv2.resize(img, (nw, nh))
    dw, dh = (640 - nw) / 2.0, (640 - nh) / 2.0
    pad = cv2.copyMakeBorder(res, int(round(dh-0.1)), int(round(dh+0.1)), int(round(dw-0.1)), int(round(dw+0.1)), cv2.BORDER_CONSTANT, value=(114,114,114))
    blob = np.transpose(cv2.cvtColor(pad, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0, (2, 0, 1))[None, ...]
    ort_out = sess.run(None, {in_meta.name: blob})[0]
    pt_boxes = model.predict(img, imgsz=640, conf=0.25, verbose=False)[0].boxes
    ort_count = int((ort_out[0][:, 4] >= 0.25).sum()) if ort_out.ndim == 3 and ort_out.shape[2] == 6 else -1
    print(f"  Parity {vp.name}: PyTorch={len(pt_boxes)} box(es) | Pure ORT={ort_count} box(es) | shape={ort_out.shape}")

display(FileLink("yolo_strips.onnx"))